# Text-to-image on a GPU

This notebook generates images from text prompts using SD-Turbo on an L40S. Everything it
needs is staged at `/shared/workshops/ood_jupyter`, so nothing downloads while you run it.

Work down the cells in order.

## 1. Install the kernel

A kernel is just the Python environment your cells actually execute in. The one Jupyter
started with does not have PyTorch in it, so the cell below registers the environment we
staged in `/shared` as a kernel you can pick.

In [ ]:
!/shared/workshops/ood_jupyter/venv/bin/python -m ipykernel install --user --name ood-demo --display-name "OOD Demo"

Now switch to it. Click where it says `Python 3` in the top right, choose `OOD Demo`, and
click Select. Then carry on with the next cell, since you do not need to run the one above
again.

## 2. Check your GPU

You asked for a GPU when you filled in the session form, so it is worth confirming you
actually got one before going any further.

In [ ]:
import subprocess
import torch

print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"],
                     capture_output=True, text=True).stdout)

print("torch", torch.__version__)
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

You should see `NVIDIA L40S-12Q` with about 12 GB of memory, which is a slice of one of the
four L40S cards in whichever vgpu node you landed on.

If `GPU available` comes back False, your session started with 0 GPUs. Everything below still
works, it will just take a minute or so per image instead of about a second.

## 3. Load the model

SD-Turbo produces an image in a single step, where the models it was distilled from need
around 50, which is why this feels as fast as it does. The weights are about 2.5 GB and are
already on disk.

Loading and generation live in `ooddemo.py`, next to the weights in `/shared`. Everything it
returns goes through the safety checker, which blanks anything it flags, so if an image comes
back black, that is why. Have a read of it if you want to see what it is doing.

In [ ]:
import sys

sys.path.insert(0, "/shared/workshops/ood_jupyter")

import ooddemo

ooddemo.load()
print("loaded on", ooddemo.device())

The first load takes 10 to 20 seconds while the weights come off Lustre.

## 4. Generate an image

Each call takes about a second once the model is loaded.

In [ ]:
import time

prompt = "a florida panther in a data center, digital art"

start = time.time()
image = ooddemo.generate(prompt)
print(f"{time.time() - start:.2f} seconds")

image

## 5. Change the prompt

Edit the text inside the quotes and press Shift+Enter to run it again. Each generation takes
about a second, so try a few. Keep prompts workplace appropriate, since anything the safety
checker flags comes back blank.

In [ ]:
prompt = "change this text to whatever you want"

ooddemo.generate(prompt)

## 6. Several at once

Because each image only takes about a second, you can put together a whole set quickly.

In [ ]:
import matplotlib.pyplot as plt

prompts = [
    "a watercolor painting of palm trees in a hurricane",
    "an oil painting of a satellite orbiting earth",
    "a photograph of a robot marine biologist",
    "a pencil sketch of a rocket launch",
]

images = [ooddemo.generate(p) for p in prompts]

fig, axes = plt.subplots(1, len(prompts), figsize=(16, 4.5))
for ax, img, p in zip(axes, images, prompts):
    ax.imshow(img)
    ax.set_title(p, fontsize=7, wrap=True)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 7. Interactive controls

The same thing again, with controls instead of editing code.

Steps trades speed for detail. The seed decides which image you get, so holding it fixed while
you change the prompt lets you compare like with like. Compare runs the same prompt and seed
at 1 step and 4 steps next to each other.

In [ ]:
import time

import ipywidgets as widgets
import matplotlib.pyplot as plt
from IPython.display import clear_output, display

prompt_box = widgets.Text(
    value="a florida panther in a data center, digital art",
    description="Prompt",
    layout=widgets.Layout(width="640px"),
)
steps_slider = widgets.IntSlider(value=1, min=1, max=4, description="Steps")
seed_box = widgets.IntText(value=42, description="Seed")
generate_button = widgets.Button(description="Generate", button_style="primary")
compare_button = widgets.Button(description="Compare 1 vs 4 steps")
output = widgets.Output()


def on_generate(_):
    with output:
        clear_output(wait=True)
        start = time.time()
        image = ooddemo.generate(prompt_box.value, steps_slider.value, seed_box.value)
        print(f"{time.time() - start:.2f} seconds at {steps_slider.value} step(s)")
        display(image)


def on_compare(_):
    with output:
        clear_output(wait=True)
        fig, axes = plt.subplots(1, 2, figsize=(9, 5))
        for ax, n in zip(axes, [1, 4]):
            start = time.time()
            ax.imshow(ooddemo.generate(prompt_box.value, n, seed_box.value))
            ax.set_title(f"{n} steps, {time.time() - start:.2f} s", fontsize=9)
            ax.axis("off")
        plt.tight_layout()
        plt.show()


generate_button.on_click(on_generate)
compare_button.on_click(on_compare)

display(widgets.VBox([
    prompt_box,
    steps_slider,
    seed_box,
    widgets.HBox([generate_button, compare_button]),
    output,
]))

## 8. Save an image

Anything you write lands in your home directory on AI.Panther, and you can pull it down to
your laptop from the Files menu in Open OnDemand.

In [ ]:
import os

image.save(os.path.expanduser("~/image.png"))
print("saved to ~/image.png")

## When you are done

Go to My Interactive Sessions and delete the session to hand the GPU back. There are only 24
slices across the three vgpu nodes, so someone else is probably waiting for it.